# ✋ Hand Gesture Recognition & Air-Writing — ZERO LAG Edition

### 🚀 How it works (why NO lag):
- **100% runs in your browser** — MediaPipe runs via CDN JavaScript, zero Python round-trips per frame
- Python is only used to display the HTML widget — all detection & drawing is pure JS at 30fps+

### Run ALL cells top to bottom, then interact with the widget below Cell 2

In [ ]:
# CELL 1 — No pip installs needed! MediaPipe runs in browser via CDN
print('✅ Ready! Run Cell 2 to launch the zero-lag interface.')

✅ Ready! Run Cell 2 to launch the zero-lag interface.


In [ ]:
# CELL 2 — Launch Zero-Lag Hand Gesture + Air-Writing System
from IPython.display import display, HTML

HTML_APP = '''
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: #0a0a12;
    font-family: "Segoe UI", Inter, sans-serif;
    color: #fff;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 16px 8px;
    gap: 12px;
  }
  h1 { font-size: 20px; color: #e0e0ff; letter-spacing: .4px; }
  h1 span { color: #5b9eff; }

  #main-layout {
    display: flex;
    gap: 16px;
    width: 100%;
    max-width: 1180px;
    align-items: flex-start;
  }

  /* ── Camera area ── */
  #cam-col {
    flex: 1;
    display: flex;
    flex-direction: column;
    gap: 8px;
  }
  #canvas-wrapper {
    position: relative;
    width: 100%;
    border-radius: 14px;
    overflow: hidden;
    border: 2.5px solid #2a6ef5;
    background: #000;
    box-shadow: 0 0 30px #1a3a8f55;
  }
  #videoEl { display: none; }
  #drawCanvas {
    position: absolute; top:0; left:0;
    width:100%; height:100%;
    pointer-events: none;
  }
  #camCanvas {
    display: block;
    width: 100%;
    height: auto;
    min-height: 460px;
  }

  /* ── Gesture overlay badge ── */
  #gesture-overlay {
    position: absolute;
    top: 12px; left: 12px;
    background: rgba(10,10,20,0.75);
    border: 1.5px solid #2a6ef5;
    border-radius: 10px;
    padding: 6px 14px;
    font-size: 16px;
    font-weight: 600;
    backdrop-filter: blur(6px);
    display: flex;
    align-items: center;
    gap: 8px;
  }
  #gesture-overlay .dot {
    width: 10px; height: 10px;
    border-radius: 50%;
    background: #555;
    transition: background .2s;
  }
  #gesture-overlay .dot.active { background: #00ff88; box-shadow: 0 0 8px #00ff88; }

  /* FPS badge */
  #fps-badge {
    position: absolute;
    top: 12px; right: 12px;
    background: rgba(10,10,20,0.7);
    border: 1px solid #333;
    border-radius: 8px;
    padding: 4px 10px;
    font-size: 12px;
    color: #0f9;
    font-family: monospace;
  }

  /* Draw mode indicator bottom-left */
  #draw-status {
    position: absolute;
    bottom: 12px; left: 12px;
    font-size: 13px;
    padding: 5px 12px;
    border-radius: 8px;
    background: rgba(0,0,0,0.6);
    border: 1px solid #333;
    transition: all .2s;
  }
  #draw-status.on { border-color: #00ff88; color: #00ff88; }
  #draw-status.off { border-color: #555; color: #888; }

  /* Buttons */
  #btn-row { display: flex; gap: 10px; }
  .btn {
    flex: 1; padding: 10px;
    border: none; border-radius: 9px;
    cursor: pointer; font-size: 14px;
    font-weight: 600; transition: all .15s;
  }
  #startBtn { background: #2a6ef5; color: #fff; }
  #startBtn:hover { background: #1a5ee0; }
  #clearBtn { background: #222; color: #ccc; border: 1.5px solid #333; }
  #clearBtn:hover { background: #333; }
  #stopBtn { background: #c0392b; color: #fff; display: none; }
  #stopBtn:hover { background: #a93226; }

  #status-bar {
    font-size: 13px; color: #7af;
    background: #12122a;
    border: 1px solid #223;
    border-radius: 8px;
    padding: 7px 14px;
    text-align: center;
  }

  /* ── Side panel ── */
  #side-panel {
    width: 250px;
    flex-shrink: 0;
    display: flex;
    flex-direction: column;
    gap: 10px;
  }
  .card {
    background: #13131f;
    border: 1px solid #22223a;
    border-radius: 12px;
    padding: 14px 15px;
  }
  .card h3 {
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 1.2px;
    color: #555;
    margin-bottom: 10px;
  }

  /* Live gesture card */
  #live-card {
    text-align: center;
    background: linear-gradient(135deg,#12122a,#1a1a35);
    border-color: #2a2a55;
  }
  #live-emoji { font-size: 52px; line-height: 1.2; transition: all .15s; }
  #live-name  { font-size: 15px; font-weight: 700; color: #adf; margin-top: 4px; }
  #live-desc  { font-size: 12px; color: #557; margin-top: 3px; }

  /* Confidence bar */
  #conf-bar-wrap { margin-top: 8px; }
  #conf-label { font-size: 11px; color: #556; margin-bottom: 3px; }
  #conf-bar-bg {
    background: #1a1a2a; border-radius: 4px; height: 7px; overflow: hidden;
  }
  #conf-bar {
    height: 100%; background: linear-gradient(90deg,#2a6ef5,#00e5ff);
    border-radius: 4px; transition: width .2s;
    width: 0%;
  }

  /* Gesture guide */
  .g-row {
    display: flex; align-items: center; gap: 10px;
    padding: 7px 0;
    border-bottom: 1px solid #1a1a2a;
    cursor: default;
    transition: background .15s;
    border-radius: 6px;
    padding-left: 4px;
  }
  .g-row:last-child { border-bottom: none; }
  .g-row:hover { background: #1a1a2e; }
  .g-row.active-gesture { background: #0d2040; border-color: #2a6ef5; }
  .g-ico { font-size: 22px; width: 30px; text-align: center; }
  .g-text .name { font-size: 13px; color: #ccc; font-weight: 600; }
  .g-text .act  { font-size: 11px; color: #558; margin-top: 1px; }

  /* Tips */
  .tip {
    font-size: 12.5px; color: #99b;
    line-height: 1.65;
    padding: 7px 10px;
    background: #0e0e1e;
    border-left: 3px solid #2a6ef5;
    border-radius: 0 7px 7px 0;
    margin-bottom: 7px;
  }
  .tip b { color: #7af; }
  .tip:last-child { margin-bottom: 0; }

  /* Stroke counter */
  #stroke-info {
    display: flex; justify-content: space-between;
    font-size: 12px; color: #556;
    margin-top: 4px;
  }
  #stroke-info span { color: #7af; font-weight: 700; }

  /* Save button */
  #saveBtn { background: #1a7a4a; color: #fff; border: 1.5px solid #2aaa6a; }
  #saveBtn:hover { background: #156040; }

  /* Canvas preview modal */
  #preview-modal {
    display: none;
    position: fixed; inset: 0;
    background: rgba(0,0,0,0.85);
    z-index: 9999;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    gap: 16px;
  }
  #preview-modal.open { display: flex; }
  #preview-box {
    background: #13131f;
    border: 2px solid #2a6ef5;
    border-radius: 16px;
    padding: 20px;
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 14px;
    max-width: 92vw;
  }
  #preview-box h2 { font-size: 17px; color: #adf; }
  #preview-img {
    border-radius: 10px;
    border: 1.5px solid #333;
    background: #000;
    max-width: 80vw;
    max-height: 60vh;
  }
  #preview-stats {
    font-size: 13px; color: #668;
    display: flex; gap: 20px;
  }
  #preview-stats span { color: #7af; font-weight: 700; }
  .modal-btns { display: flex; gap: 10px; }
  .modal-btns button {
    padding: 9px 22px; border-radius: 8px; border: none;
    cursor: pointer; font-size: 14px; font-weight: 600;
  }
  #downloadBtn { background: #2a6ef5; color: #fff; }
  #downloadBtn:hover { background: #1a5ee0; }
  #closeModalBtn { background: #222; color: #ccc; border: 1.5px solid #333; }
  #closeModalBtn:hover { background: #333; }
</style>
</head>
<body>

<h1>✋ Hand Gesture Recognition &amp; <span>Air-Writing</span></h1>

<div id="main-layout">

  <!-- LEFT: Camera -->
  <div id="cam-col">
    <div id="canvas-wrapper">
      <video id="videoEl" autoplay playsinline muted></video>
      <canvas id="camCanvas"></canvas>
      <canvas id="drawCanvas"></canvas>
      <div id="gesture-overlay">
        <span class="dot" id="dot"></span>
        <span id="gest-text">Waiting...</span>
      </div>
      <div id="fps-badge">-- fps</div>
      <div id="draw-status" class="off">○ Not drawing</div>
    </div>
    <div id="status-bar">Click "Start Camera" to begin</div>
    <div id="btn-row">
      <button class="btn" id="startBtn" onclick="startCamera()">▶ Start Camera</button>
      <button class="btn" id="clearBtn" onclick="clearCanvas()">🗑 Clear</button>
      <button class="btn" id="stopBtn"  onclick="stopCamera()">⏹ Stop</button>
    </div>
    <div id="stroke-info">
      <span>Strokes: <span id="stroke-count">0</span></span>
      <span>Points drawn: <span id="point-count">0</span></span>
    </div>
  </div>

  <!-- RIGHT: Side Panel -->
  <div id="side-panel">

    <!-- Live gesture card -->
    <div class="card" id="live-card">
      <h3>Current Gesture</h3>
      <div id="live-emoji">👋</div>
      <div id="live-name">No hand detected</div>
      <div id="live-desc">Show your hand to the camera</div>
      <div id="conf-bar-wrap">
        <div id="conf-label">Confidence: 0%</div>
        <div id="conf-bar-bg"><div id="conf-bar"></div></div>
      </div>
    </div>

    <!-- Gesture guide -->
    <div class="card">
      <h3>Gesture Guide</h3>
      <div class="g-row" id="row-Pointing">
        <span class="g-ico">☝️</span>
        <div class="g-text"><div class="name">Index Finger</div><div class="act">✍ Draw on canvas</div></div>
      </div>
      <div class="g-row" id="row-Fist">
        <span class="g-ico">✊</span>
        <div class="g-text"><div class="name">Fist</div><div class="act">⏸ Lift pen / pause</div></div>
      </div>
      <div class="g-row" id="row-ThumbsUp">
        <span class="g-ico">👍</span>
        <div class="g-text"><div class="name">Thumbs Up</div><div class="act">🗑 Clear entire canvas</div></div>
      </div>
      <div class="g-row" id="row-OpenHand">
        <span class="g-ico">🖐</span>
        <div class="g-text"><div class="name">Open Hand</div><div class="act">⏸ Pause drawing</div></div>
      </div>
      <div class="g-row" id="row-Peace">
        <span class="g-ico">✌️</span>
        <div class="g-text"><div class="name">Peace / V-Sign</div><div class="act">👁 Detected &amp; shown</div></div>
      </div>
      <div class="g-row" id="row-RockOn">
        <span class="g-ico">🤘</span>
        <div class="g-text"><div class="name">Rock On</div><div class="act">👁 Detected &amp; shown</div></div>
      </div>
      <div class="g-row" id="row-OK">
        <span class="g-ico">👌</span>
        <div class="g-text"><div class="name">OK Sign</div><div class="act">👁 Detected &amp; shown</div></div>
      </div>
      <div class="g-row" id="row-ThumbsDown">
        <span class="g-ico">👎</span>
        <div class="g-text"><div class="name">Thumbs Down</div><div class="act">👁 Detected &amp; shown</div></div>
      </div>
    </div>

    <!-- Tips -->
    <div class="card">
      <h3>Tips for Best Results</h3>
      <div class="tip"><b>💡 Lighting</b><br>Face a bright lamp or window. Avoid backlight.</div>
      <div class="tip"><b>📏 Distance</b><br>Keep hand 30–60 cm away. Whole hand visible.</div>
      <div class="tip"><b>✍ Writing</b><br>Move slowly. Use ✊ fist to lift pen between letters.</div>
      <div class="tip"><b>🧹 Erase</b><br>Hold 👍 thumbs up to wipe canvas clean.</div>
    </div>

  </div><!-- end side panel -->
</div><!-- end main layout -->

<!-- Save/Preview Modal -->
<div id=\"preview-modal\">
  <div id=\"preview-box\">
    <h2>🖊 Your Air-Written Canvas</h2>
    <canvas id=\"preview-img\"></canvas>
    <div id=\"preview-stats\">
      Strokes: <span id=\"modal-strokes\">0</span>
      &nbsp;&nbsp;Points: <span id=\"modal-points\">0</span>
      &nbsp;&nbsp;Size: <span id=\"modal-size\">--</span>
    </div>
    <div class=\"modal-btns\">
      <button id=\"downloadBtn\" onclick=\"downloadDrawing()\">⬇️ Download PNG</button>
      <button id=\"closeModalBtn\" onclick=\"closePreview()\">✕ Close</button>
    </div>
  </div>
</div>

<!-- MediaPipe via CDN -->
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/control_utils/control_utils.js" crossorigin="anonymous"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>

<script>
// ──────────────────────────────────────────────────────────────────
// GESTURE DEFINITIONS
// ──────────────────────────────────────────────────────────────────
const GESTURES = {
  Pointing:   { emoji: "☝️",  name: "Pointing",    desc: "Drawing mode active!", draw: true  },
  Fist:       { emoji: "✊",  name: "Fist",         desc: "Pen lifted — paused",  draw: false },
  ThumbsUp:   { emoji: "👍",  name: "Thumbs Up",    desc: "Canvas cleared!",      draw: false },
  ThumbsDown: { emoji: "👎",  name: "Thumbs Down",  desc: "Detected",             draw: false },
  OpenHand:   { emoji: "🖐",  name: "Open Hand",    desc: "All 5 fingers open",   draw: false },
  Peace:      { emoji: "✌️",  name: "Peace / V",    desc: "Two fingers up",       draw: false },
  RockOn:     { emoji: "🤘",  name: "Rock On",      desc: "Index + Pinky up",     draw: false },
  OK:         { emoji: "👌",  name: "OK Sign",      desc: "Thumb + Index circle", draw: false },
  Unknown:    { emoji: "🤚",  name: "Hand Detected",desc: "Unclassified pose",    draw: false },
  None:       { emoji: "👋",  name: "No hand",      desc: "Show hand to camera",  draw: false },
};

// ──────────────────────────────────────────────────────────────────
// GESTURE CLASSIFICATION
// ──────────────────────────────────────────────────────────────────
function classifyGesture(lm) {
  // lm = array of 21 {x,y,z} landmarks
  // Finger tip and pip landmark indices
  const tips = [4, 8, 12, 16, 20];
  const pips = [3, 6, 10, 14, 18]; // proximal for fingers 1-4, use different for thumb
  const mcps = [2, 5,  9, 13, 17];

  // Is a finger extended? tip.y < pip.y (in image coords, y increases downward)
  const ext = [
    lm[4].x < lm[3].x,            // thumb: tip left of knuckle (mirrored)
    lm[8].y  < lm[6].y,           // index
    lm[12].y < lm[10].y,          // middle
    lm[16].y < lm[14].y,          // ring
    lm[20].y < lm[18].y,          // pinky
  ];

  const [thumb, index, middle, ring, pinky] = ext;
  const count = ext.slice(1).filter(Boolean).length; // fingers excl thumb

  // ── Thumb up/down (only thumb, rest curled) ──
  if (thumb && !index && !middle && !ring && !pinky) {
    // thumb direction: tip above wrist = thumbs up
    if (lm[4].y < lm[0].y) return "ThumbsUp";
    else                    return "ThumbsDown";
  }

  // ── Fist ──
  if (!thumb && !index && !middle && !ring && !pinky) return "Fist";

  // ── Open hand ──
  if (index && middle && ring && pinky) return "OpenHand";

  // ── Pointing (only index) ──
  if (index && !middle && !ring && !pinky) return "Pointing";

  // ── Peace / V-sign ──
  if (index && middle && !ring && !pinky) return "Peace";

  // ── Rock on (index + pinky) ──
  if (index && !middle && !ring && pinky) return "RockOn";

  // ── OK sign: thumb tip near index tip ──
  const thumbIndexDist = Math.hypot(lm[4].x - lm[8].x, lm[4].y - lm[8].y);
  if (thumbIndexDist < 0.07 && middle && ring && pinky) return "OK";

  return "Unknown";
}

// ──────────────────────────────────────────────────────────────────
// CANVAS & DRAWING
// ──────────────────────────────────────────────────────────────────
const videoEl  = document.getElementById("videoEl");
const camCvs   = document.getElementById("camCanvas");
const drawCvs  = document.getElementById("drawCanvas");
const camCtx   = camCvs.getContext("2d");
const drawCtx  = drawCvs.getContext("2d");

let isDrawing    = false;
let lastPt       = null;
let smoothPts    = [];       // smoothing buffer
const SMOOTH_N   = 6;
let strokeCount  = 0;
let pointCount   = 0;
let lastGesture  = "None";
let activeRow    = null;

// FPS tracking
let fpsFrames = 0, fpsLast = performance.now();

function resizeCanvases(w, h) {
  camCvs.width  = drawCvs.width  = w;
  camCvs.height = drawCvs.height = h;
  // Restore draw canvas settings after resize
  drawCtx.lineCap   = "round";
  drawCtx.lineJoin  = "round";
}

function clearCanvas() {
  drawCtx.clearRect(0, 0, drawCvs.width, drawCvs.height);
  strokeCount = 0; pointCount = 0;
  updateStrokeInfo();
}

function updateStrokeInfo() {
  document.getElementById("stroke-count").textContent = strokeCount;
  document.getElementById("point-count").textContent  = pointCount;
}

// Smooth a point using rolling average
function getSmoothedPoint(x, y) {
  smoothPts.push({x, y});
  if (smoothPts.length > SMOOTH_N) smoothPts.shift();
  const sx = smoothPts.reduce((a,p) => a+p.x, 0) / smoothPts.length;
  const sy = smoothPts.reduce((a,p) => a+p.y, 0) / smoothPts.length;
  return {x: sx, y: sy};
}

function drawPoint(x, y, startNew) {
  const sp = getSmoothedPoint(x, y);
  if (startNew || !lastPt) {
    drawCtx.beginPath();
    drawCtx.moveTo(sp.x, sp.y);
    strokeCount++;
  } else {
    // Glow: wide faint line first
    drawCtx.globalAlpha = 0.18;
    drawCtx.strokeStyle = "#00ffaa";
    drawCtx.lineWidth   = 16;
    drawCtx.beginPath();
    drawCtx.moveTo(lastPt.x, lastPt.y);
    drawCtx.lineTo(sp.x, sp.y);
    drawCtx.stroke();

    // Main line
    drawCtx.globalAlpha = 1.0;
    drawCtx.strokeStyle = "#00ffaa";
    drawCtx.lineWidth   = 4;
    drawCtx.beginPath();
    drawCtx.moveTo(lastPt.x, lastPt.y);
    drawCtx.lineTo(sp.x, sp.y);
    drawCtx.stroke();
  }
  lastPt = sp;
  pointCount++;
  updateStrokeInfo();
}

// ──────────────────────────────────────────────────────────────────
// UI UPDATES
// ──────────────────────────────────────────────────────────────────
function updateGestureUI(key, confidence) {
  const g = GESTURES[key] || GESTURES.Unknown;
  document.getElementById("live-emoji").textContent = g.emoji;
  document.getElementById("live-name").textContent  = g.name;
  document.getElementById("live-desc").textContent  = g.desc;

  const pct = Math.round((confidence || 1) * 100);
  document.getElementById("conf-label").textContent = `Confidence: ${pct}%`;
  document.getElementById("conf-bar").style.width   = pct + "%";

  // Gesture badge on video
  document.getElementById("gest-text").textContent = g.emoji + " " + g.name;
  const dot = document.getElementById("dot");
  dot.className = "dot" + (g.draw ? " active" : "");

  // Draw status
  const ds = document.getElementById("draw-status");
  if (g.draw) {
    ds.textContent = "● Drawing";
    ds.className   = "on";
  } else {
    ds.textContent = "○ Not drawing";
    ds.className   = "off";
  }

  // Highlight active gesture row
  if (activeRow) activeRow.classList.remove("active-gesture");
  const row = document.getElementById("row-" + key);
  if (row) { row.classList.add("active-gesture"); activeRow = row; }
}

// ──────────────────────────────────────────────────────────────────
// MEDIAPIPE SETUP
// ──────────────────────────────────────────────────────────────────
let mpCamera = null;
let mpHands  = null;
let stream   = null;

function drawSkeleton(lm, W, H) {
  // Connections from MediaPipe hand model
  const connections = [
    [0,1],[1,2],[2,3],[3,4],
    [0,5],[5,6],[6,7],[7,8],
    [0,9],[9,10],[10,11],[11,12],
    [0,13],[13,14],[14,15],[15,16],
    [0,17],[17,18],[18,19],[19,20],
    [5,9],[9,13],[13,17]
  ];
  camCtx.save();
  // Bones
  camCtx.strokeStyle = "rgba(80,180,255,0.85)";
  camCtx.lineWidth   = 2;
  camCtx.lineCap     = "round";
  connections.forEach(([a,b]) => {
    camCtx.beginPath();
    camCtx.moveTo(lm[a].x * W, lm[a].y * H);
    camCtx.lineTo(lm[b].x * W, lm[b].y * H);
    camCtx.stroke();
  });
  // Joints
  lm.forEach((p, i) => {
    const isTip = [4,8,12,16,20].includes(i);
    camCtx.beginPath();
    camCtx.arc(p.x * W, p.y * H, isTip ? 6 : 4, 0, Math.PI*2);
    camCtx.fillStyle   = isTip ? "#00ffaa" : "#4ab4ff";
    camCtx.fill();
    camCtx.strokeStyle = "#fff";
    camCtx.lineWidth   = 1.5;
    camCtx.stroke();
  });
  // Index fingertip glow when drawing
  if (isDrawing) {
    camCtx.beginPath();
    camCtx.arc(lm[8].x*W, lm[8].y*H, 16, 0, Math.PI*2);
    const grad = camCtx.createRadialGradient(
      lm[8].x*W, lm[8].y*H, 0,
      lm[8].x*W, lm[8].y*H, 16
    );
    grad.addColorStop(0,   "rgba(0,255,170,0.45)");
    grad.addColorStop(1,   "rgba(0,255,170,0)");
    camCtx.fillStyle = grad;
    camCtx.fill();
  }
  camCtx.restore();
}

function onResults(results) {
  const W = camCvs.width, H = camCvs.height;

  // Draw mirrored video frame
  camCtx.save();
  camCtx.translate(W, 0);
  camCtx.scale(-1, 1);
  camCtx.drawImage(results.image, 0, 0, W, H);
  camCtx.restore();

  // Overlay the draw canvas (air writing)
  camCtx.drawImage(drawCvs, 0, 0);

  // FPS
  fpsFrames++;
  const now = performance.now();
  if (now - fpsLast >= 1000) {
    document.getElementById("fps-badge").textContent = fpsFrames + " fps";
    fpsFrames = 0; fpsLast = now;
  }

  if (!results.multiHandLandmarks || results.multiHandLandmarks.length === 0) {
    updateGestureUI("None", 0);
    isDrawing = false; lastPt = null; smoothPts = [];
    return;
  }

  const lm      = results.multiHandLandmarks[0];
  const gesture = classifyGesture(lm);
  const conf    = results.multiHandWorldLandmarks ? 0.95 : 0.85;

  // Mirror landmarks for drawing (since we flipped video)
  const mirrorX = (v) => (1 - v) * W;

  // Mirror landmark array for skeleton drawing
  const mirrorLm = lm.map(p => ({ x: 1-p.x, y: p.y, z: p.z }));
  drawSkeleton(mirrorLm, W, H);

  // Update gesture UI
  updateGestureUI(gesture, conf);

  // Handle actions
  if (gesture === "ThumbsUp") {
    clearCanvas();
  }

  // Air writing
  const gInfo = GESTURES[gesture];
  if (gInfo && gInfo.draw) {
    const ix = mirrorX(lm[8].x);
    const iy = lm[8].y * H;
    const startNew = !isDrawing;
    isDrawing = true;
    drawPoint(ix, iy, startNew);
  } else {
    if (isDrawing) { isDrawing = false; lastPt = null; smoothPts = []; }
  }
}

// ──────────────────────────────────────────────────────────────────
// CAMERA CONTROL
// ──────────────────────────────────────────────────────────────────
async function startCamera() {
  document.getElementById("status-bar").textContent = "📷 Requesting camera...";

  try {
    stream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 1280 }, height: { ideal: 720 }, frameRate: { ideal: 30 } }
    });
  } catch(e) {
    document.getElementById("status-bar").textContent = "❌ Camera denied: " + e.message;
    return;
  }

  videoEl.srcObject = stream;
  await videoEl.play();

  const W = videoEl.videoWidth  || 1280;
  const H = videoEl.videoHeight || 720;
  resizeCanvases(W, H);

  // Init MediaPipe Hands
  mpHands = new Hands({ locateFile: (f) =>
    `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${f}` });

  mpHands.setOptions({
    maxNumHands: 1,
    modelComplexity: 1,
    minDetectionConfidence: 0.7,
    minTrackingConfidence: 0.6,
  });
  mpHands.onResults(onResults);

  // MediaPipe Camera loop
  mpCamera = new Camera(videoEl, {
    onFrame: async () => { await mpHands.send({ image: videoEl }); },
    width: W, height: H
  });
  await mpCamera.start();

  document.getElementById("status-bar").textContent = "✅ Running — show your hand!";
  document.getElementById("startBtn").style.display = "none";
  document.getElementById("stopBtn").style.display  = "block";
}

function stopCamera() {
  if (mpCamera) mpCamera.stop();
  if (stream)   stream.getTracks().forEach(t => t.stop());
  document.getElementById("status-bar").textContent = "⏹ Camera stopped.";
  document.getElementById("startBtn").style.display = "block";
  document.getElementById("stopBtn").style.display  = "none";
  updateGestureUI("None", 0);
}
// ──────────────────────────────────────────────────────────────────
// SAVE / PREVIEW DRAWING
// ──────────────────────────────────────────────────────────────────
function saveDrawing() {
  const W = drawCvs.width, H = drawCvs.height;

  // Check if canvas has anything drawn
  const imgData = drawCtx.getImageData(0, 0, W, H).data;
  const hasContent = imgData.some((v, i) => i % 4 === 3 && v > 0); // check alpha channel

  if (!hasContent) {
    document.getElementById(\"status-bar\").textContent = \"⚠️ Canvas is empty — draw something first!\";
    setTimeout(() => {
      document.getElementById(\"status-bar\").textContent = \"✅ Running — show your hand!\";
    }, 2500);
    return;
  }

  // Compose: black background + draw strokes
  const previewCvs = document.getElementById(\"preview-img\");
  previewCvs.width  = W;
  previewCvs.height = H;
  const pCtx = previewCvs.getContext(\"2d\");

  // Dark background
  pCtx.fillStyle = \"#0a0a12\";
  pCtx.fillRect(0, 0, W, H);

  // Draw the strokes
  pCtx.drawImage(drawCvs, 0, 0);

  // Watermark
  pCtx.fillStyle = \"rgba(90,150,255,0.25)\";
  pCtx.font = \"14px Inter, sans-serif\";
  pCtx.fillText(\"Air-Writing System\", 12, H - 12);

  // Stats
  document.getElementById(\"modal-strokes\").textContent = strokeCount;
  document.getElementById(\"modal-points\").textContent  = pointCount;
  document.getElementById(\"modal-size\").textContent    = W + \" × \" + H;

  // Show modal
  document.getElementById(\"preview-modal\").classList.add(\"open\");
}

function downloadDrawing() {
  const previewCvs = document.getElementById(\"preview-img\");
  const link = document.createElement(\"a\");
  link.download = \"air-writing-\" + new Date().toISOString().slice(0,19).replace(/:/g,\"-\") + \".png\";
  link.href = previewCvs.toDataURL(\"image/png\");
  link.click();
}

function closePreview() {
  document.getElementById(\"preview-modal\").classList.remove(\"open\");
}

// Close modal on backdrop click
document.getElementById(\"preview-modal\").addEventListener(\"click\", (e) => {
  if (e.target.id === \"preview-modal\") closePreview();
});
</script>
</body></html>
'''

display(HTML(HTML_APP))
print('🚀 Interface launched! Click "Start Camera" above and allow browser camera access.')
print('💡 No pip install needed — MediaPipe runs 100% in your browser at 30fps+')

🚀 Interface launched! Click "Start Camera" above and allow browser camera access.
💡 No pip install needed — MediaPipe runs 100% in your browser at 30fps+


In [ ]:
# ============================================================
# CELL 4 (Optional) — Test with a Static Image
# Run this if you don't have a webcam or want to test offline
# ============================================================
from google.colab import files
import io

print('Upload a photo of a hand gesture:')
uploaded = files.upload()

for filename, data in uploaded.items():
    pil  = Image.open(io.BytesIO(data)).convert('RGB')
    frame = np.array(pil)
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    with mp_hands.Hands(static_image_mode=True,
                        min_detection_confidence=0.5) as h:
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = h.process(rgb)

    gesture = 'No hand detected'
    if results.multi_hand_landmarks:
        for lm_set in results.multi_hand_landmarks:
            gesture = classify_gesture(lm_set.landmark)
            mp_drawing.draw_landmarks(
                frame, lm_set, mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

    cv2.putText(frame, gesture, (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)

    out_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 6))
    plt.imshow(out_rgb)
    plt.axis('off')
    plt.title(f'Detected Gesture: {gesture}', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f'\n🎯 Result: {gesture}')

Upload a photo of a hand gesture:
